In [1]:
!pip install fastapi uvicorn asynci


ERROR: Could not find a version that satisfies the requirement asynci (from versions: none)
ERROR: No matching distribution found for asynci


In [2]:
!pip install redis
!apt-get install -y redis-server > /dev/null
!redis-server --daemonize yes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.8/409.8 kB 8.2 MB/s eta 0:00:00


In [3]:
import redis
import uuid
import json
import time
import threading
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()
# Подключение к Redis
r = redis.Redis(host='localhost', port=6379, decode_responses=True)

class ApplicationRequest(BaseModel):
    client_name: str
    amount: float
    ssn: str   # Идентификатор

@app.post("/api/v1/apply")
def apply(request: ApplicationRequest):  # НЕ async!
    application_id = str(uuid.uuid4())

    #  Формируем событие
    event = {
        "event_type": "ApplicationCreated",
        "data": {
            "application_id": application_id,
            "client_name": request.client_name,
            "amount": request.amount,
            "ssn": request.ssn
        }
    }

      # Публикуем в Redis канал «fintech_events»
    r.publish("fintech_events", json.dumps(event))

    r.hset(f"app:{application_id}", "status", "processing")
       #  Возвращаем пользователю ID сразу, не ждём скоринга
    return {"status": "processing", "application_id": application_id}

@app.get("/api/v1/status/{app_id}")
def get_status(app_id: str):
    # Здесь Gateway должен сходить в свою БД (или кэш), куда Deal Service скинул финальный статус
    # Для упрощения можно хранить статусы в Redis
    status = r.hget(f"app:{app_id}", "status")
    return {"status": status or "unknown"}

# SCORING (как в твоем стиле)
def scoring():
    print("Scoring service запущен")
    result = r.blpop(['fintech_events'], timeout=2)
    if result:
        data = json.loads(result[1])
        print("Скоринг", data['data']['application_id'])
        #  Имитация сложных расчетов (ML модели и т.д.)
        time.sleep(8)
        # Логика скоринга:
        score = 0.85 if data['data']['amount'] < 10000 else 0.45
        r.publish("fintech_events", json.dumps({
            "event_type": "ScoreCalculated",
            "data": {**data['data'], "score": score}
        }))
        print("Score:", score)

# DEAL
def deal():
    print("Deal service запущен")
    result = r.blpop(['fintech_events'], timeout=12)
    if result:
        data = json.loads(result[1])
        app_id = data['data']['application_id']
        time.sleep(3)
        status = "approved" if data['data']['score'] > 0.7 else "rejected"
        r.hset(f"app:{app_id}", "status", status)
        print(app_id, "->", status)

if __name__ == "__main__":
    t1 = threading.Thread(target=scoring)
    t2 = threading.Thread(target=deal)
    t1.start(); t2.start()

    time.sleep(1)
    resp = apply(ApplicationRequest(client_name="Иван", amount=5000, ssn="123"))
    print("Запрос:", resp)

    time.sleep(12)
    print("Финал:", r.hget(f"app:{resp['application_id']}", "status"))

Scoring service запущен
Deal service запущен
Запрос: {'status': 'processing', 'application_id': '3a1ae7c9-47d5-4483-8731-7e3910086d11'}
Финал: processing


In [4]:
import os
import redis
import uuid
import json
import time
import threading
import asyncio
import pandas as pd
from IPython.display import display, HTML
from fastapi import FastAPI
from pydantic import BaseModel

# 1. Запуск Redis
!apt-get install -y redis-server > /dev/null
!redis-server --daemonize yes

app = FastAPI()
r = redis.Redis(host='localhost', port=6379, decode_responses=True)

#  API Gateway

@app.get("/api/v1/dashboard/{ssn}")
async def get_dashboard(ssn: str):
    async def get_client_info():
        await asyncio.sleep(0.1)
        return r.hgetall(f"client_profile:{ssn}") or {"status": "No profile"}

    async def get_active_deals():
        await asyncio.sleep(0.1)
        keys = r.keys(f"app:*")
        deals = []
        for k in keys:
            data = r.hgetall(k)
            if data.get('ssn') == ssn:
                deals.append({"id": k.replace('app:', ''), "status": data.get('status')})
        return deals

    client_info, deals = await asyncio.gather(get_client_info(), get_active_deals())
    return {"ssn": ssn, "profile": client_info, "active_deals": deals}

#  Services

def scoring_service():
    while True:
        result = r.brpop(['fintech_queue'], timeout=2)
        if result:
            data = json.loads(result[1])['data']
            ssn = data['ssn']
            r.hset(f"client_profile:{ssn}", mapping={"name": data['client_name'], "last_update": time.strftime('%H:%M:%S')})
            time.sleep(0.5)
            score = 0.85 if data['amount'] < 10000 else 0.45
            r.lpush("deal_queue", json.dumps({**data, "score": score}))

def deal_service():
    while True:
        result = r.brpop(['deal_queue'], timeout=2)
        if result:
            data = json.loads(result[1])
            app_id = data['application_id']
            if data.get('ssn') == '999':
                r.lpush("saga_compensation_queue", json.dumps({"app_id": app_id, "reason": "Blacklisted SSN"}))
                r.hset(f"app:{app_id}", mapping={"status": "failed", "ssn": data['ssn']})
            else:
                status = "approved" if data['score'] > 0.7 else "rejected"
                r.hset(f"app:{app_id}", mapping={"status": status, "ssn": data['ssn']})

def saga_compensator():
    while True:
        result = r.brpop(['saga_compensation_queue'], timeout=2)
        if result:
            data = json.loads(result[1])
            r.hset(f"app:{data['app_id']}", "status", f"compensation_applied: {data['reason']}")

#  Execution
if __name__ == "__main__":
    r.flushall()
    for func in [scoring_service, deal_service, saga_compensator]:
        threading.Thread(target=func, daemon=True).start()

    test_cases = [
        {"client_name": "Alice", "amount": 5000, "ssn": "101"},   # Approved
        {"client_name": "Bob", "amount": 15000, "ssn": "102"},    # Rejected (low score)
        {"client_name": "Charlie", "amount": 2000, "ssn": "999"}, # Saga (debt)
        {"client_name": "David", "amount": 8000, "ssn": "104"},   # Approved
        {"client_name": "Eve", "amount": 12000, "ssn": "105"}     # Rejected (low score)
    ]

    for case in test_cases:
        app_id = str(uuid.uuid4())
        r.lpush("fintech_queue", json.dumps({"data": {**case, "application_id": app_id}}))

    print("Processing 5 applications...")
    time.sleep(7)

    print("\n--- Dashboard Results ---")
    for ssn in ["101", "102", "999", "104", "105"]:
        dash = await get_dashboard(ssn)
        display(HTML(f"<hr><h3>Dashboard SSN: {ssn}</h3>"))
        display(pd.DataFrame([dash['profile']]))
        if dash['active_deals']:
            display(pd.DataFrame(dash['active_deals']))

Processing 5 applications...

--- Dashboard Results ---


,name,last_update
0,Alice,03:03:54


,id,status
0,b9a1036e-b6e5-4e30-aa64-6d74e9d7ffb7,approved


,name,last_update
0,Bob,03:03:55


,id,status
0,c539d13e-ccf2-42e1-8a6f-d02955273e11,rejected


...

,id,status
0,bce168b8-544e-4cf0-80b0-f4f2a0b73046,failed


,name,last_update
0,David,03:03:56


,id,status
0,df9fd3e5-8d78-4594-9817-de55e4d2c489,approved


,name,last_update
0,Eve,03:03:56


,id,status
0,03933595-9b50-4397-91be-6c0b1f795e70,rejected


Принцип работы:

API Gateway: Асинхронно собирает данные из разных 'хранилищ' (Redis).
Saga (Choreography): Если Deal Service видит проблему (SSN 999), он не просто останавливается, а создает событие компенсации, на которое реагирует Saga Compensator, откатывая статус.
Event-driven: Сервисы общаются через очереди Redis (brpop/lpush), что обеспечивает асинхронную обработку.
Соответствие заданию:

Добавлен асинхронный эндпоинт /dashboard.
Реализована компенсация Saga через событие DealCreationFailed.
Все данные агрегируются из имитированных read-моделей.


1. Консистентность данных (Eventual Consistency):

В FinTech это стандарт для некритичных операций (обновление профиля, маркетинговые уведомления).
Например,  в нашем коде get_dashboard может вернуть статус processing, хотя Scoring уже завершен, но Deal еще не успел обработать очередь. Это не критично для интерфейса пользователя. Критично это при списании средств: нельзя допустить 'двойной траты', там все еще нужны распределенные транзакции или строгий порядок событий.

2. Отказоустойчивость:

Потеря заявок: В нашей реализации с Redis (lpush / brpop) 0 заявок будет потеряно.
Ибо Redis выступает буфером. Если Scoring Service упадет, сообщения будут копиться в списке fintech_queue. Как только сервис поднимется, он начнет очередь с того же места. Это преимущество асинхронности перед REST, где при падении сервиса клиент сразу получил бы ошибку 500.

3. Наблюдаемость (Correlation ID):

В нашем коде мы уже используем application_id. Это и есть Correlation ID.
Его нужно передавать в заголовках каждого сообщения и логгировать: logger.info(f"[{app_id}] Scoring started"). Это позволяет склеить логи разных сервисов в одну цепочку.

4. API Gateway (3 vs 4):

В нашем примере Gateway — умный, так как он сам идет в Redis, собирает данные из разных ключей и агрегирует их в один JSON для /dashboard. «Тупой» Gateway просто перенаправил бы запрос на один сервис.
Gateway становится «распределенным монолитом», если в нем начинает оседать бизнес-логика (например, расчет скоринга прямо в Gateway). Это делает его слишком сложным и мешает независимой разработке сервисов.